# ECSC Developmental Analysis v2: Age Effects

**Changes from v1:**
- Lower word threshold (200 words) to reduce selection bias
- Adaptive target size (10% of doc, min 20 tokens)
- Focus on SHORT context effects (4-32 tokens)
- Group by actual age, not study year

**Metrics:**
1. Perplexity at minimal context (baseline unpredictability)
2. Early slope (perplexity drop 4→32 tokens)
3. Short-range half-life
4. Total benefit from context

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import files

print("Upload transcripts.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

In [ ]:
# Load data with LOWER threshold
MIN_WORDS = 200  # Down from 400!

records = []
with open(DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        record['pop'] = json.loads(record['population'])
        record['word_count'] = len(record['text'].split())
        record['age_months'] = record['pop']['age_months']
        records.append(record)

df_all = pd.DataFrame(records)
df = df_all[df_all['word_count'] >= MIN_WORDS].copy()

# Age bins
def age_bin(age):
    if age < 72:  # <6yr
        return '4-6yr'
    elif age < 96:  # 6-8yr
        return '6-8yr'
    elif age < 120:  # 8-10yr
        return '8-10yr'
    else:  # 10+yr
        return '10+yr'

df['age_group'] = df['age_months'].apply(age_bin)

print(f"Total documents: {len(df_all)}")
print(f"After filter (>={MIN_WORDS} words): {len(df)} ({100*len(df)/len(df_all):.0f}%)")
print(f"\nBy age group:")
for ag in ['4-6yr', '6-8yr', '8-10yr', '10+yr']:
    n = len(df[df['age_group'] == ag])
    ages = df[df['age_group'] == ag]['age_months']
    if n > 0:
        print(f"  {ag}: n={n}, mean age={ages.mean():.1f} mo")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens in [target_start, target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    return np.exp(total_loss / count), total_loss / count


def analyze_document_adaptive(text, min_target=20, target_fraction=0.10):
    """
    Adaptive context ablation:
    - Target size = max(min_target, 10% of doc)
    - Context lengths scaled to available tokens
    """
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    # Adaptive target size
    target_size = max(min_target, int(n_tokens * target_fraction))
    target_size = min(target_size, n_tokens - 8)  # Leave room for some context
    
    if target_size < min_target:
        return [], {}
    
    max_context = n_tokens - target_size
    
    # Dense sampling at short contexts, sparse at long
    context_lengths = []
    # Every 4 tokens up to 32
    context_lengths.extend([c for c in range(4, min(33, max_context+1), 4)])
    # Every 16 tokens from 48 to 128
    context_lengths.extend([c for c in range(48, min(129, max_context+1), 16)])
    # Every 32 tokens beyond
    context_lengths.extend([c for c in range(160, max_context+1, 32)])
    
    context_lengths = sorted(set(context_lengths))
    
    if len(context_lengths) < 3:
        return [], {}
    
    results = []
    
    for ctx_len in context_lengths:
        doc_start = n_tokens - target_size - ctx_len
        truncated_tokens = full_tokens[doc_start:]
        
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        
        results.append({
            'context_length': ctx_len,
            'perplexity': ppl,
            'loss': loss,
        })
    
    meta = {
        'n_tokens': n_tokens,
        'target_size': target_size,
        'max_context': max_context,
    }
    
    return results, meta

In [ ]:
# Run analysis
all_results = []
doc_metrics = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    doc_results, meta = analyze_document_adaptive(row['text'])
    
    if not doc_results or len(doc_results) < 3:
        continue
    
    # Store raw results
    for r in doc_results:
        r['doc_id'] = row['doc_id']
        r['age_months'] = row['age_months']
        r['age_group'] = row['age_group']
        r['word_count'] = row['word_count']
        all_results.append(r)
    
    # Compute document metrics
    df_doc = pd.DataFrame(doc_results).sort_values('context_length')
    contexts = df_doc['context_length'].values
    ppls = df_doc['perplexity'].values
    
    # Metric 1: Perplexity at minimal context (4 tokens)
    ppl_min_ctx = ppls[0]
    
    # Metric 2: Perplexity at 32 tokens (if available)
    ppl_32 = df_doc[df_doc['context_length'] == 32]['perplexity'].values
    ppl_32 = ppl_32[0] if len(ppl_32) > 0 else np.nan
    
    # Metric 3: Early drop (4 -> 32 tokens)
    if not np.isnan(ppl_32):
        early_drop = ppl_min_ctx - ppl_32
        early_drop_pct = 100 * early_drop / ppl_min_ctx
    else:
        early_drop = np.nan
        early_drop_pct = np.nan
    
    # Metric 4: Half-life (standard)
    total_benefit = ppls[0] - ppls[-1]
    half_life = np.nan
    if total_benefit > 0:
        target_ppl = ppls[0] - 0.5 * total_benefit
        for i in range(len(ppls) - 1):
            if ppls[i] >= target_ppl >= ppls[i+1]:
                frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    # Metric 5: Minimum perplexity (at max context)
    ppl_max_ctx = ppls[-1]
    
    # Metric 6: Early slope (linear fit 4-32 tokens)
    early_mask = contexts <= 32
    if early_mask.sum() >= 3:
        slope, intercept, r, p, se = stats.linregress(contexts[early_mask], ppls[early_mask])
        early_slope = slope  # Negative = perplexity decreasing (good)
    else:
        early_slope = np.nan
    
    doc_metrics.append({
        'doc_id': row['doc_id'],
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        'word_count': row['word_count'],
        'n_tokens': meta['n_tokens'],
        'target_size': meta['target_size'],
        'ppl_min_ctx': ppl_min_ctx,
        'ppl_32': ppl_32,
        'ppl_max_ctx': ppl_max_ctx,
        'early_drop': early_drop,
        'early_drop_pct': early_drop_pct,
        'early_slope': early_slope,
        'half_life': half_life,
        'total_benefit': total_benefit,
    })

results_df = pd.DataFrame(all_results)
metrics_df = pd.DataFrame(doc_metrics)

print(f"\nProcessed {len(metrics_df)} documents")
print(f"By age group: {metrics_df['age_group'].value_counts().to_dict()}")

In [ ]:
print("=" * 70)
print("METRICS BY AGE GROUP")
print("=" * 70)

metrics_to_test = [
    ('ppl_min_ctx', 'Perplexity @ 4 tokens', 'higher = less predictable'),
    ('ppl_32', 'Perplexity @ 32 tokens', 'higher = less predictable'),
    ('early_drop', 'Early drop (4→32)', 'higher = more local benefit'),
    ('early_drop_pct', 'Early drop %', 'higher = more relative local benefit'),
    ('early_slope', 'Early slope', 'more negative = steeper drop'),
    ('half_life', 'Half-life', 'higher = needs more context'),
    ('ppl_max_ctx', 'Min perplexity', 'lower = more predictable overall'),
]

age_groups = ['4-6yr', '6-8yr', '8-10yr', '10+yr']
age_groups = [ag for ag in age_groups if len(metrics_df[metrics_df['age_group'] == ag]) > 0]

summary_results = []

for metric, name, interpretation in metrics_to_test:
    print(f"\n{name} ({interpretation}):")
    print("-" * 50)
    
    group_data = []
    for ag in age_groups:
        vals = metrics_df[metrics_df['age_group'] == ag][metric].dropna()
        if len(vals) > 0:
            print(f"  {ag}: n={len(vals)}, mean={vals.mean():.2f}, std={vals.std():.2f}")
            group_data.append(vals.values)
    
    if len(group_data) >= 2:
        h, p = stats.kruskal(*group_data)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"  Kruskal-Wallis: H={h:.2f}, p={p:.4f} {sig}")
        
        # Correlation with continuous age
        valid = metrics_df[[metric, 'age_months']].dropna()
        if len(valid) > 10:
            r, p_corr = stats.spearmanr(valid['age_months'], valid[metric])
            sig_corr = "*" if p_corr < 0.05 else ""
            print(f"  Age correlation: rho={r:.3f}, p={p_corr:.4f} {sig_corr}")
            
            summary_results.append({
                'metric': name,
                'kruskal_p': p,
                'spearman_rho': r,
                'spearman_p': p_corr
            })

In [ ]:
print("\n" + "=" * 70)
print("SUMMARY: Which metrics show age effects?")
print("=" * 70)

summary_df = pd.DataFrame(summary_results)
summary_df['kruskal_sig'] = summary_df['kruskal_p'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
summary_df['spearman_sig'] = summary_df['spearman_p'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')

print(summary_df[['metric', 'kruskal_p', 'kruskal_sig', 'spearman_rho', 'spearman_p', 'spearman_sig']].to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

colors = {'4-6yr': '#e74c3c', '6-8yr': '#f39c12', '8-10yr': '#27ae60', '10+yr': '#3498db'}

# 1. Perplexity curves by age
ax = axes[0, 0]
for ag in age_groups:
    ag_df = results_df[results_df['age_group'] == ag]
    means = ag_df.groupby('context_length')['perplexity'].mean()
    ax.plot(means.index, means.values, marker='o', label=ag, color=colors.get(ag, 'gray'), markersize=4)
ax.set_xlabel('Context Length')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity Curves by Age')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Early perplexity (@ 4 tokens) by age
ax = axes[0, 1]
data = [metrics_df[metrics_df['age_group'] == ag]['ppl_min_ctx'].dropna() for ag in age_groups]
bp = ax.boxplot(data, labels=age_groups, patch_artist=True)
for patch, ag in zip(bp['boxes'], age_groups):
    patch.set_facecolor(colors.get(ag, 'gray'))
    patch.set_alpha(0.7)
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity @ 4 tokens (minimal context)')
ax.grid(True, alpha=0.3, axis='y')

# 3. Early slope by age
ax = axes[0, 2]
data = [metrics_df[metrics_df['age_group'] == ag]['early_slope'].dropna() for ag in age_groups]
bp = ax.boxplot(data, labels=age_groups, patch_artist=True)
for patch, ag in zip(bp['boxes'], age_groups):
    patch.set_facecolor(colors.get(ag, 'gray'))
    patch.set_alpha(0.7)
ax.set_ylabel('Slope (ppl/token)')
ax.set_title('Early Slope (4-32 tokens)')
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3, axis='y')

# 4. Scatter: age vs ppl_min_ctx
ax = axes[1, 0]
for ag in age_groups:
    ag_data = metrics_df[metrics_df['age_group'] == ag]
    ax.scatter(ag_data['age_months'], ag_data['ppl_min_ctx'], alpha=0.6, label=ag, color=colors.get(ag, 'gray'))
ax.set_xlabel('Age (months)')
ax.set_ylabel('Perplexity @ 4 tokens')
ax.set_title('Age vs Minimal-Context Perplexity')
# Add trend line
valid = metrics_df[['age_months', 'ppl_min_ctx']].dropna()
z = np.polyfit(valid['age_months'], valid['ppl_min_ctx'], 1)
p = np.poly1d(z)
ax.plot(valid['age_months'].sort_values(), p(valid['age_months'].sort_values()), 'k--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Scatter: age vs early_slope
ax = axes[1, 1]
for ag in age_groups:
    ag_data = metrics_df[metrics_df['age_group'] == ag]
    ax.scatter(ag_data['age_months'], ag_data['early_slope'], alpha=0.6, label=ag, color=colors.get(ag, 'gray'))
ax.set_xlabel('Age (months)')
ax.set_ylabel('Early Slope')
ax.set_title('Age vs Early Slope')
valid = metrics_df[['age_months', 'early_slope']].dropna()
z = np.polyfit(valid['age_months'], valid['early_slope'], 1)
p = np.poly1d(z)
ax.plot(valid['age_months'].sort_values(), p(valid['age_months'].sort_values()), 'k--', alpha=0.5)
ax.axhline(0, color='red', linestyle='--', alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Early drop % by age
ax = axes[1, 2]
data = [metrics_df[metrics_df['age_group'] == ag]['early_drop_pct'].dropna() for ag in age_groups]
bp = ax.boxplot(data, labels=age_groups, patch_artist=True)
for patch, ag in zip(bp['boxes'], age_groups):
    patch.set_facecolor(colors.get(ag, 'gray'))
    patch.set_alpha(0.7)
ax.set_ylabel('% Drop')
ax.set_title('% Perplexity Drop (4→32 tokens)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ecsc_age_analysis_v2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairwise comparisons for significant metrics
print("=" * 70)
print("PAIRWISE COMPARISONS FOR KEY METRICS")
print("=" * 70)

for metric in ['ppl_min_ctx', 'early_slope', 'early_drop_pct']:
    print(f"\n{metric}:")
    for i, ag1 in enumerate(age_groups):
        for ag2 in age_groups[i+1:]:
            v1 = metrics_df[metrics_df['age_group'] == ag1][metric].dropna().values
            v2 = metrics_df[metrics_df['age_group'] == ag2][metric].dropna().values
            if len(v1) >= 5 and len(v2) >= 5:
                stat, p = stats.mannwhitneyu(v1, v2, alternative='two-sided')
                d = (np.mean(v1) - np.mean(v2)) / np.sqrt((np.var(v1) + np.var(v2)) / 2)
                sig = "*" if p < 0.05 else ""
                print(f"  {ag1} vs {ag2}: diff={np.mean(v1)-np.mean(v2):.2f}, d={d:.2f}, p={p:.4f} {sig}")

In [ ]:
# Save results
results_df.to_csv('ecsc_age_v2_raw.csv', index=False)
metrics_df.to_csv('ecsc_age_v2_metrics.csv', index=False)

print("Saved:")
print("  - ecsc_age_v2_raw.csv")
print("  - ecsc_age_v2_metrics.csv")
print("  - ecsc_age_analysis_v2.png")

In [ ]:
from google.colab import files
files.download('ecsc_age_v2_raw.csv')
files.download('ecsc_age_v2_metrics.csv')
files.download('ecsc_age_analysis_v2.png')

## Interpretation

**Metrics to watch:**

1. **Perplexity @ minimal context**: How predictable is the text with almost no context?
   - Higher for older kids might mean more complex/varied language
   - Lower might mean more formulaic

2. **Early slope**: How quickly does perplexity drop in first 32 tokens?
   - Steeper (more negative) = text benefits more from local context
   - Age effect could go either way

3. **Early drop %**: What fraction of total benefit comes from first 32 tokens?
   - Higher = more local coherence
   - Lower = benefits spread across more context

**Sample size is key** - with 200+ word threshold, we should have much better coverage of younger children.